# Contrastive Learning with Evaluation and Embedding Export



In [ ]:
!pip -q install -U torch transformers pandas numpy selfies pyarrow fastparquet

In [ ]:
import os
import math
import time
import json
import random
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import selfies as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

WORKDIR = Path('/content/morpheus')
if not WORKDIR.exists():
    raise FileNotFoundError('Clone your repository to /content/morpheus before running this notebook.')

In [ ]:
class ChemicalTokenizer:
    def __init__(self, tokenizer_path: str):
        with open(tokenizer_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        self.token_to_id = data['token_to_id']
        self.id_to_token = {int(k): v for k, v in data['id_to_token'].items()}
        self.pad_token_id = int(data['pad_token_id'])
        self.mask_token_id = int(data['mask_token_id'])
        self.eos_token_id = int(data['eos_token_id'])
        self.vocab_size = int(data['vocab_size'])

    def encode(self, selfies_str: str) -> List[int]:
        tokens = list(sf.split_selfies(selfies_str))
        return [self.token_to_id.get(t, self.pad_token_id) for t in tokens]


class TimestepEmbedding(nn.Module):
    def __init__(self, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )

    def sinusoidal_features(self, t: torch.Tensor) -> torch.Tensor:
        device = t.device
        half = self.hidden_size // 2
        frequencies = torch.exp(
            torch.arange(half, device=device) * -(math.log(10000.0) / max(half - 1, 1))
        )
        angles = t * frequencies.unsqueeze(0)
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        features = self.sinusoidal_features(t)
        return self.mlp(features)


class PositionalEmbedding(nn.Module):
    def __init__(self, max_length: int, hidden_size: int):
        super().__init__()
        self.embedding = nn.Embedding(max_length, hidden_size)

    def forward(self, seq_len: int, device: torch.device) -> torch.Tensor:
        positions = torch.arange(seq_len, device=device)
        return self.embedding(positions).unsqueeze(0)


class SelfAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        bsz, seq_len, _ = x.shape

        q = self.q_proj(x).view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_mask = None
        if padding_mask is not None:
            attn_mask = torch.zeros((bsz, 1, 1, seq_len), dtype=x.dtype, device=x.device)
            mask_bool = padding_mask.unsqueeze(1).unsqueeze(2)
            attn_mask = attn_mask.masked_fill(mask_bool, float('-inf'))

        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.dropout.p if self.training else 0.0,
        )

        attended = attended.transpose(1, 2).contiguous().view(bsz, seq_len, self.hidden_size)
        return self.out_proj(attended)


class FeedForward(nn.Module):
    def __init__(self, hidden_size: int, ffn_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_dim),
            nn.SiLU(),
            nn.Linear(ffn_dim, hidden_size),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, ffn_dim: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attention = SelfAttention(hidden_size, num_heads, dropout)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = FeedForward(hidden_size, ffn_dim, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        residual = x
        x = self.norm1(x)
        x = self.attention(x, padding_mask)
        x = self.dropout(x)
        x = residual + x

        residual = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = residual + x
        return x


class MolecularDiffusionModel(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_heads, ffn_dim, num_layers, max_length, pad_token_id, dropout=0.1):
        super().__init__()
        self.pad_token_id = pad_token_id
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.token_embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.pos_embedding = PositionalEmbedding(max_length, hidden_size)
        self.timestep_embedding = TimestepEmbedding(hidden_size)
        self.input_norm = nn.LayerNorm(hidden_size)
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_size, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])
        self.output_norm = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

        self._init_weights()
        self.lm_head.weight = self.token_embedding.weight

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()

    def forward(self, input_ids: torch.Tensor, timesteps: torch.Tensor) -> torch.Tensor:
        device = input_ids.device
        bsz, seq_len = input_ids.shape
        padding_mask = (input_ids == self.pad_token_id)

        x = self.token_embedding(input_ids)
        x = x + self.pos_embedding(seq_len, device)
        x = x + self.timestep_embedding(timesteps).unsqueeze(1)
        x = self.input_norm(x)
        for block in self.blocks:
            x = block(x, padding_mask)
        x = self.output_norm(x)
        return self.lm_head(x)


def resolve_device(device: str) -> torch.device:
    device = device.lower().strip()
    if device != 'auto':
        return torch.device(device)
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def mean_pool_last_hidden(last_hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    masked = last_hidden * mask
    summed = masked.sum(dim=1)
    denom = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / denom


def load_text_encoder(model_name: str, device: torch.device):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    return tokenizer, model


def load_molecule_backbone(checkpoint_path: Path, tokenizer_path: Path, device: torch.device):
    chem_tokenizer = ChemicalTokenizer(str(tokenizer_path))
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = checkpoint['config']

    model = MolecularDiffusionModel(
        vocab_size=config['vocab_size'],
        hidden_size=config['hidden_size'],
        num_heads=config['num_heads'],
        ffn_dim=config['ffn_dim'],
        num_layers=config['num_layers'],
        max_length=config['max_length'],
        pad_token_id=chem_tokenizer.pad_token_id,
        dropout=0.0,
    ).to(device)

    model.load_state_dict(checkpoint['model'])
    return chem_tokenizer, model, config


def encode_selfies_batch(selfies_list: Sequence[str], chem_tokenizer: ChemicalTokenizer, max_length: int, device: torch.device) -> torch.Tensor:
    batch_ids = []
    for s in selfies_list:
        ids = chem_tokenizer.encode(s)
        ids = ids[:max_length - 1]
        ids.append(chem_tokenizer.eos_token_id)
        if len(ids) < max_length:
            ids += [chem_tokenizer.pad_token_id] * (max_length - len(ids))
        batch_ids.append(ids)
    return torch.tensor(batch_ids, dtype=torch.long, device=device)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]]):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]

def read_pairs_from_csv(csv_path: Path, text_column: str, selfies_column: str, max_rows: Optional[int] = None):
    df = pd.read_csv(csv_path)
    if max_rows is not None:
        df = df.head(max_rows).copy()
    pairs = []
    for _, row in df.iterrows():
        text = str(row[text_column]) if pd.notna(row[text_column]) else ''
        selfies = str(row[selfies_column]) if pd.notna(row[selfies_column]) else ''
        if text.strip() and selfies.strip():
            pairs.append((text, selfies))
    return pairs

def train_val_split(pairs, val_fraction=0.1, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(pairs))
    rng.shuffle(idx)
    split = int(len(pairs) * (1.0 - val_fraction))
    train_pairs = [pairs[i] for i in idx[:split]]
    val_pairs = [pairs[i] for i in idx[split:]]
    return train_pairs, val_pairs

class MoleculeEncoder(nn.Module):
    def __init__(self, molecular_model: MolecularDiffusionModel):
        super().__init__()
        self.molecular_model = molecular_model

    def forward(self, input_ids: torch.Tensor, pad_token_id: int) -> torch.Tensor:
        device = input_ids.device
        batch_size, seq_len = input_ids.shape
        padding_mask = (input_ids == pad_token_id)
        attention_mask = (~padding_mask).long()

        x = self.molecular_model.token_embedding(input_ids)
        x = x + self.molecular_model.pos_embedding(seq_len, device)
        t = torch.zeros((batch_size, 1), device=device)
        x = x + self.molecular_model.timestep_embedding(t).unsqueeze(1)
        x = self.molecular_model.input_norm(x)
        for block in self.molecular_model.blocks:
            x = block(x, padding_mask)
        x = self.molecular_model.output_norm(x)
        return mean_pool_last_hidden(x, attention_mask)

class ContrastiveAligner(nn.Module):
    def __init__(self, text_model, molecule_encoder, text_hidden, mol_hidden, proj_dim):
        super().__init__()
        self.text_model = text_model
        self.molecule_encoder = molecule_encoder
        self.text_proj = nn.Linear(text_hidden, proj_dim)
        self.mol_proj = nn.Linear(mol_hidden, proj_dim)

    def encode_text(self, text_inputs):
        out = self.text_model(**text_inputs)
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            pooled = mean_pool_last_hidden(out.last_hidden_state, text_inputs['attention_mask'])
        return F.normalize(self.text_proj(pooled), dim=-1)

    def encode_molecule(self, mol_ids, pad_token_id):
        pooled = self.molecule_encoder(mol_ids, pad_token_id)
        return F.normalize(self.mol_proj(pooled), dim=-1)

    def forward(self, text_inputs, mol_ids, pad_token_id):
        z_text = self.encode_text(text_inputs)
        z_mol = self.encode_molecule(mol_ids, pad_token_id)
        return z_text, z_mol

def contrastive_loss(z_text, z_mol, temperature=0.07):
    logits = (z_text @ z_mol.T) / temperature
    labels = torch.arange(logits.shape[0], device=logits.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

def retrieval_metrics(z_text, z_mol, ks=(1, 5, 10)):
    sims = z_text @ z_mol.T
    sorted_idx = torch.argsort(sims, dim=1, descending=True)
    targets = torch.arange(sims.shape[0], device=sims.device)
    m = {}
    for k in ks:
        topk = sorted_idx[:, :k]
        m[f'recall@{k}'] = (topk == targets.unsqueeze(1)).any(dim=1).float().mean().item()
    sorted_np = sorted_idx.detach().cpu().numpy()
    ranks = []
    rr = []
    for i in range(sorted_np.shape[0]):
        r = int(np.where(sorted_np[i] == i)[0][0]) + 1
        ranks.append(r)
        rr.append(1.0 / r)
    m['mean_rank'] = float(np.mean(ranks))
    m['mrr'] = float(np.mean(rr))
    return m

In [ ]:
CFG = {
    'input_csv': WORKDIR / 'data' / 'text_selfies_pairs.csv',  
    'text_column': 'prompt',
    'selfies_column': 'selfies',
    'molecular_checkpoint': WORKDIR / 'checkpoints' / 'best_model.pt',
    'chemical_tokenizer': WORKDIR / 'chemical_tokenizer.json',
    'text_model': 'BAAI/bge-large-en-v1.5',
    'batch_size': 64,
    'epochs': 5,
    'lr': 2e-5,
    'weight_decay': 1e-2,
    'temperature': 0.07,
    'proj_dim': 768,
    'max_text_length': 256,
    'val_fraction': 0.1,
    'seed': 42,
    'max_rows': None,
    'freeze_text': False,
    'freeze_molecule': False,
    'device': 'cuda',
    'output_checkpoint': WORKDIR / 'checkpoints' / 'contrastive_text_selfies.pt'
}

device = resolve_device(CFG['device'])
print('Device:', device)

pairs = read_pairs_from_csv(CFG['input_csv'], CFG['text_column'], CFG['selfies_column'], CFG['max_rows'])
print('Valid pairs:', len(pairs))
train_pairs, val_pairs = train_val_split(pairs, val_fraction=CFG['val_fraction'], seed=CFG['seed'])
print('Train/Val:', len(train_pairs), len(val_pairs))

text_tokenizer, text_model = load_text_encoder(CFG['text_model'], device)
chem_tokenizer, molecular_model, mol_config = load_molecule_backbone(
    CFG['molecular_checkpoint'], CFG['chemical_tokenizer'], device
)

model = ContrastiveAligner(
    text_model=text_model,
    molecule_encoder=MoleculeEncoder(molecular_model),
    text_hidden=int(text_model.config.hidden_size),
    mol_hidden=int(mol_config['hidden_size']),
    proj_dim=CFG['proj_dim']
).to(device)

if CFG['freeze_text']:
    for p in model.text_model.parameters():
        p.requires_grad = False
if CFG['freeze_molecule']:
    for p in model.molecule_encoder.parameters():
        p.requires_grad = False

In [ ]:
# Train
def make_collate_fn(tokenizer, max_text_length, device):
    def collate(batch):
        texts = [x[0] for x in batch]
        selfies = [x[1] for x in batch]
        text_inputs = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_text_length,
            return_tensors='pt',
        )
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
        return text_inputs, selfies
    return collate

train_loader = DataLoader(
    PairDataset(train_pairs),
    batch_size=CFG['batch_size'],
    shuffle=True,
    drop_last=True,
    collate_fn=make_collate_fn(text_tokenizer, CFG['max_text_length'], device),
)
val_loader = DataLoader(
    PairDataset(val_pairs),
    batch_size=CFG['batch_size'],
    shuffle=False,
    drop_last=False,
    collate_fn=make_collate_fn(text_tokenizer, CFG['max_text_length'], device),
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
)

history = []
best_val_loss = float('inf')

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    tr_loss_sum = 0.0
    tr_steps = 0

    for text_inputs, selfies in train_loader:
        mol_ids = encode_selfies_batch(selfies, chem_tokenizer, int(mol_config['max_length']), device)
        z_text, z_mol = model(text_inputs, mol_ids, chem_tokenizer.pad_token_id)
        loss = contrastive_loss(z_text, z_mol, CFG['temperature'])

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        tr_loss_sum += float(loss.item())
        tr_steps += 1

    train_loss = tr_loss_sum / max(tr_steps, 1)

    model.eval()
    val_loss_sum = 0.0
    val_steps = 0
    val_z_text = []
    val_z_mol = []

    with torch.no_grad():
        for text_inputs, selfies in val_loader:
            mol_ids = encode_selfies_batch(selfies, chem_tokenizer, int(mol_config['max_length']), device)
            z_text, z_mol = model(text_inputs, mol_ids, chem_tokenizer.pad_token_id)
            loss = contrastive_loss(z_text, z_mol, CFG['temperature'])
            val_loss_sum += float(loss.item())
            val_steps += 1
            val_z_text.append(z_text)
            val_z_mol.append(z_mol)

    val_loss = val_loss_sum / max(val_steps, 1)
    val_metrics = retrieval_metrics(torch.cat(val_z_text, dim=0), torch.cat(val_z_mol, dim=0), ks=(1, 5, 10))

    rec = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, **val_metrics}
    history.append(rec)

    print(
        f"Epoch {epoch}/{CFG['epochs']} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
        f"R@1={val_metrics['recall@1']:.4f} R@5={val_metrics['recall@5']:.4f} "
        f"R@10={val_metrics['recall@10']:.4f} MRR={val_metrics['mrr']:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        CFG['output_checkpoint'].parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            'epoch': epoch,
            'config': CFG,
            'molecular_config': mol_config,
            'model_state': model.state_dict(),
            'best_val_loss': best_val_loss,
            'history': history,
        }, CFG['output_checkpoint'])
        print('Saved best checkpoint:', CFG['output_checkpoint'])

hist_path = CFG['output_checkpoint'].with_suffix('.history.json')
with open(hist_path, 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2, default=str)
print('Saved history:', hist_path)

In [ ]:
# Eval
EVAL_CSV = CFG['input_csv']
EVAL_MAX_ROWS = None

pairs_eval = read_pairs_from_csv(EVAL_CSV, CFG['text_column'], CFG['selfies_column'], EVAL_MAX_ROWS)
eval_loader = DataLoader(
    PairDataset(pairs_eval),
    batch_size=min(128, CFG['batch_size'] * 2),
    shuffle=False,
    drop_last=False,
    collate_fn=make_collate_fn(text_tokenizer, CFG['max_text_length'], device),
)

model.eval()
all_z_text = []
all_z_mol = []
with torch.no_grad():
    for text_inputs, selfies in eval_loader:
        mol_ids = encode_selfies_batch(selfies, chem_tokenizer, int(mol_config['max_length']), device)
        zt, zm = model(text_inputs, mol_ids, chem_tokenizer.pad_token_id)
        all_z_text.append(zt)
        all_z_mol.append(zm)

metrics = retrieval_metrics(torch.cat(all_z_text, dim=0), torch.cat(all_z_mol, dim=0), ks=(1, 5, 10))
print('Contrastive Evaluation Metrics')
for k in ['recall@1', 'recall@5', 'recall@10', 'mrr', 'mean_rank']:
    print(f'  {k}: {metrics[k]:.6f}')

eval_json = WORKDIR / 'outputs' / 'contrastive_eval_metrics.json'
eval_json.parent.mkdir(parents=True, exist_ok=True)
with open(eval_json, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics to:', eval_json)

In [ ]:
# Export embeddings
PROMPT_CSV = WORKDIR / 'data' / 'prompts.csv'  
PROMPT_COLUMN = 'prompt'
PROMPT_OUT = WORKDIR / 'outputs' / 'prompt_embeddings_bge_large.parquet'
TIME_BUDGET_MINUTES = 60
SAFETY_FACTOR = 0.90
SAMPLE_ROWS = 4096
BATCH_SIZE = 128
MAX_LENGTH = 256
TEXT_MODEL = 'BAAI/bge-large-en-v1.5'

text_tok, text_encoder = load_text_encoder(TEXT_MODEL, device)
text_encoder.eval()

def encode_text_batch(texts, tokenizer, model, device, max_length):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            pooled = mean_pool_last_hidden(out.last_hidden_state, enc['attention_mask'])
    return pooled.detach().cpu().numpy()

prompt_df = pd.read_csv(PROMPT_CSV)
valid = prompt_df[PROMPT_COLUMN].dropna().astype(str)
valid = valid[valid.str.strip() != '']
sample_texts = valid.head(SAMPLE_ROWS).tolist()

t0 = time.time()
done = 0
for s in range(0, len(sample_texts), BATCH_SIZE):
    _ = encode_text_batch(sample_texts[s:s+BATCH_SIZE], text_tok, text_encoder, device, MAX_LENGTH)
    done += len(sample_texts[s:s+BATCH_SIZE])
rows_per_sec = done / max(time.time() - t0, 1e-9)

target_rows = int(rows_per_sec * TIME_BUDGET_MINUTES * 60 * SAFETY_FACTOR)
max_rows = min(max(target_rows, 1), len(prompt_df))
print(f'Estimated rows/sec: {rows_per_sec:.2f}')
print(f'Planned rows in budget: {max_rows}/{len(prompt_df)}')

run_df = prompt_df.head(max_rows).copy()
embed_dim = None
matrix = []
for s in range(0, len(run_df), BATCH_SIZE):
    texts = run_df.iloc[s:s+BATCH_SIZE][PROMPT_COLUMN].fillna('').astype(str).tolist()
    emb = encode_text_batch(texts, text_tok, text_encoder, device, MAX_LENGTH)
    matrix.append(emb)

emb_matrix = np.concatenate(matrix, axis=0).astype(np.float32)
embed_dim = emb_matrix.shape[1]

out = pd.DataFrame()
out['source_index'] = run_df.index
out['text'] = run_df[PROMPT_COLUMN].fillna('').astype(str)
out['text_length'] = out['text'].str.len()
out['model_name'] = TEXT_MODEL
out['pooling'] = 'mean_pooling_or_pooler'
out['embedding_dim'] = embed_dim
out['generated_at_utc'] = datetime.now(timezone.utc).isoformat()
for i in range(embed_dim):
    out[f'emb_{i:04d}'] = emb_matrix[:, i]

PROMPT_OUT.parent.mkdir(parents=True, exist_ok=True)
try:
    out.to_parquet(PROMPT_OUT, index=False, engine='pyarrow')
    engine = 'pyarrow'
except Exception:
    out.to_parquet(PROMPT_OUT, index=False, engine='fastparquet')
    engine = 'fastparquet'

print('Saved prompt embeddings:', PROMPT_OUT)
print('Parquet engine:', engine)
print('Rows exported:', len(out))